# LPHY1365: Météorologie


## Chapitre 6: Rappels et notions complémentaires de mécanique

Source: *Triplet, J.-P., & Roche, G.* (1996). Météorologie générale. Service central de la communication et de la commercialisation de Météo-France.


Le point de départ de cette première session est une image représentant la péninsule ibérique le 17 septembre 2024, alors que le centre du Portugal est soumis à des feux de forêt d'une ampleur considérable.


Mais cette photo est-elle bien réelle? Alors que les intelligences artificielles génératives inondent le web de contenu créé de toutes pièces, pourquoi cette image ne serait-elle pas, elle aussi, une construction virtuelle? 

*Questions pour les étudiant.e.s:*
- Quels sont les indices qui permettraient de vérifier l'authenticité de l'image ou de démontrer son inauthenticité?
- D'où provient l'image? 
- La source est-elle fiable?
- Les trajectoires suivies par la fumée de combustion sont-elles alignées aux vents du jour?

*Ressources disponibles:*

https://earth.nullschool.net/

https://www.windy.com/?50.842,4.438,5

https://www.tropicaltidbits.com/

https://www.wetterzentrale.de/

https://charts.ecmwf.int/




<!--Chargement de l'image satellite montrant les feux au Portugal le 17 septembre 2024-->


![image](téléchargement.jpeg)

Source: https://atmosphere.copernicus.eu/wildfires-rage-across-northern-portugal-cams-tracks-their-impacts (consultée la dernière fois le 11 février 2026)

In [33]:
# Chargement du fichier netcdf portant le doux nom de "617b904f234593921f174fa06a177d56.nc" qui a été téléchargé depuis
# le site de la réanalyse ERA5. Importation des bibliothèques nécessaires pour manipuler les données et les visualiser.
# Ensuite, plot du champ de vent à 10 mètres d'altitude pour 12.00UTC au 17 septembre 2024.

import xarray as xr
import matplotlib.pyplot as plt
# Charger le fichier netcdf
ds = xr.open_dataset('./data/617b904f234593921f174fa06a177d56.nc')
# Afficher les variables disponibles pour vérifier que nous avons les données de vent à 10 mètres
print(ds)

# Prendre le temps d'examiner le nom des variables, dont la variables temporelle, pour s'assurer que nous sélectionnons les bonnes données. 
# Imprimer les dimensions du dataset pour comprendre la structure des données et identifier les variables de vent à 10 mètres.
print(ds.dims)


<xarray.Dataset>
Dimensions:     (valid_time: 24, latitude: 721, longitude: 1440)
Coordinates:
    number      int64 ...
  * valid_time  (valid_time) datetime64[ns] 2024-09-17 ... 2024-09-17T23:00:00
  * latitude    (latitude) float64 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    expver      (valid_time) object ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 ...
    v10         (valid_time, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-02-08T13:54 GRIB to CDM+CF via cfgrib-0.9.1...
Frozen({'valid_time': 24, 'latitude': 721, 'longitude': 1440})


In [34]:
# La dimension "temps" s'appelle donc "valid_time". Nous allons sélectionner les données de vent à 10 mètres d'altitude pour le 17 septembre 2024 à 12.00UTC.
# Sélectionner les données de vent à 10 mètres d'altitude pour le 17 septembre 2024 à 12.00UTC

# Convertir la date et l'heure en format compatible avec xarray
time_selection = '2024-09-17T12:00:00'
# Sélectionner les données de vent à 10 mètres d'altitude pour le temps spécifié
u10 = ds['u10'].sel(valid_time=time_selection)
v10 = ds['v10'].sel(valid_time=time_selection)
# Afficher les données sélectionnées pour vérifier qu'elles sont correctes
print(u10)
print(v10)
plt.show()

<xarray.DataArray 'u10' (latitude: 721, longitude: 1440)>
[1038240 values with dtype=float32]
Coordinates:
    number      int64 ...
    valid_time  datetime64[ns] 2024-09-17T12:00:00
  * latitude    (latitude) float64 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    expver      object ...
Attributes: (12/32)
    GRIB_paramId:                             165
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      1038240
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_units:                               m s**-1
    long_name:                                10 metre U wind component
    units:                                    m s**-1
    standard_name:        

In [ ]:
# Plot du champ de vent à 10 mètres par des flèches (quiver plot) pour visualiser la direction du vent.
# Nous choisissons en plus de créer un champ scalaire "wind speed" qui est la norme du vent. Les flèches
# auront une longueur identique et le fond de carte sera coloré en fonction de la vitesse du vent.
import numpy as np
# Calculer la vitesse du vent à partir des composantes u et v
wind_speed = np.sqrt(u10**2 + v10**2)
# Créer une grille de coordonnées pour le plot
lon = ds['longitude']
lat = ds['latitude']
# Créer un quiver plot pour les composantes du vent
plt.figure(figsize=(10, 6))
plt.quiver(lon, lat, u10, v10, wind_speed, scale=500, cmap='viridis')
plt.colorbar(label='Wind Speed (m/s)')
plt.title('Wind at 10m on 17 September 2024 at 12:00 UTC')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

In [ ]:
# Tout ça a une bonne tête. Faison maintenant un "zoom" sur le Portugal, qui est la zone qui nous intéresse.
# Définir les limites de la zone d'intérêt (Portugal)
lon_min, lon_max = -20, -5
lat_min, lat_max = 35, 45

# Créer un jeu de données "zoomé" pour la zone d'intérêt
u10_zoom = u10.sel(longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max))
v10_zoom = v10.sel(longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max))
wind_speed_zoom = wind_speed.sel(longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max))
lon_zoom = lon.sel(longitude=slice(lon_min, lon_max))
lat_zoom = lat.sel(latitude=slice(lat_min, lat_max))

# Créer un quiver plot pour la zone d'intérêt. Nous utilisont maintenant Cartopy pour ajouter une carte de fond.
# Importation de la lbriéthèque Cartopy pour la visualisation cartographique
plt.figure(figsize=(10, 6))
import cartopy.crs as ccrs

# Création du fond de carte avec Cartopy
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
# Ajouter les flèches de vent
plt.contourf(lon, lat, wind_speed, lonlat=True, cmap='viridis')
# normalize to get equal length arrows. plot arrows in dark grey and plot only every second arrow to avoid overcrowding
declutter = 2
plt.quiver(lon[ ::declutter], lat[::declutter], u10[::declutter, ::declutter] / wind_speed[::declutter, ::declutter], v10[::declutter, ::declutter] / wind_speed[::declutter, ::declutter], scale=50, color='darkgrey')
plt.colorbar(label='Wind Speed (m/s)')
plt.title('Wind at 10m on 17 September 2024 at 12:00 UTC (Zoom on Portugal)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()


### 6.1 - Quelques éléments de cinématique


Considérons une particule **ponctuelle** et intéressons-nous à son mouvement, décrit par une courbe $C$.

In [ ]:
# Création d'un graphe bi-dimensionnel (axes x et y) décrivant la trajectoire d'une particule M (choisie ici comme une courbe paramétrée par le temps x(t) = t et y(t) = (t-1)^2 + 1
import numpy as np
import matplotlib.pyplot as plt
# Définir la fonction de la trajectoire
def trajectory(t):
    x = t
    y = (t - 1)**3 +2
    return x, y
# Générer des points de temps pour tracer la trajectoire
t_values = np.linspace(0, 4, 100)
x_values, y_values = trajectory(t_values)

# Tracer la trajectoire
plt.figure(figsize=(6, 4), dpi = 200)
plt.plot(x_values, y_values, label='Trajectoire de M')
plt.title('Trajectoire d\'une particule $M$')
plt.xlabel('x(t)')
plt.ylabel('y(t)')
plt.xlim(-1, 3)
plt.ylim(0, 5)  
plt.grid()
plt.show()



En tant qu'objet géométrique, cette trajectoire nous renseigne certes sur l'ensemble des points de l'espace qui ont été occupés par la particule lors de son mouvement, MAIS ne nous enseigne rien sur la façon dont elle l'a parcourue! C'est pourquoi la trajectoire est mathématiquement exprimée de façon paramétrique par une variable indépendante - le temps - qui va relier entre elles les variables spatiales $x$ et $y$.

#### 6.1.1 - Le vecteur position

Le vecteur position $\vec{r}(t)$ relie une origine $0$ préalablement fixée (par exemple, le point $(0, 0)$) et le point correspondant à la position de la particule au temps $t$:

$$\vec{r}(t) := \overrightarrow{OM} $$

Ce vecteur, bien qu'essentiel pour définir les vecteurs vitesse et accélération, est en pratique rarement utilisé en physique des fluides géophysiques car l'état dynamique de l'atmosphère peut se décrire par le champ de vitesse seul, et/ou par des fonctions de celui-ci.

#### 6.1.2 - Le vecteur vitesse
On appelle $M_0$ le point de la trajectoire correspondant au temps initial $t=0$. 

Le vecteur *vitesse moyenne* entre deux instants successifs est défini comme étant la différence entre le vecteur position à deux instants successifs $t$ et $t+\Delta t$ (correspondant à deux points $M$ et $M_1$), et divisé par l'intervalle de temps qui s'est écoulé entre ces deux instants. Puisque le vecteur position est toujours défini depuis une origine commune $O$, il est facile de montrer que le vecteur vitesse moyenne est colinéaire au segment reliant M et M_1

$$ \overrightarrow{v}_m(t) = \frac{\overrightarrow{MM}_1}{\Delta t} $$

On définit le vecteur *vitesse de la particule à l'instant $t$* comme la limite du vecteur vitesse moyenne lorsque celle-ci est prise sur un intervalle qui tend vers 0:

$$\overrightarrow{v}(t) = \lim_{\Delta t \to 0} \frac{\overrightarrow{MM}_1}{\Delta t}$$


Le vecteur vitesse est donc toujours tangent à la trajectoire au point $M$, de même sens que le vecteur vitesse moyenne, et de module égal à

$$v(t) := \| \overrightarrow{v}(t)  \| = \lim_{\Delta t \to 0} \frac{\|\overrightarrow{MM}_1\|}{\Delta t}$$

In [ ]:
# Ajout sur le graphe du point M0 correspondant à t=0, et ajouter sur le graphe un point arbitraire (x,y)
# (choisi ici comme correspondant à t = 1)
t0 = 0
M0 = trajectory(t0)
tt = 1.5 # pour désigner la variable courante t
ttpD = 3.5 # Pour tt + delta t
M = trajectory(tt)
# Créer une animation avec interaction pour faire varier dt et voir le point M1 se déplacer le long de la trajectoire
from ipywidgets import interact
import ipywidgets as widgets

def update(dt):
    t1 = tt + dt
    M1 = trajectory(t1)
    plt.figure(figsize=(6, 4), dpi = 200)
    plt.plot(x_values, y_values, label='Trajectoire de M')
    plt.scatter(*M0, 100, color='red',  label=f'M0 (t=0)')
    plt.scatter(*M, 100, color='blue',  label=f'M (t={tt})', zorder = 5)
    plt.scatter(*M1, 100,  color='green', label=f'M1 (t={t1:.2f})', zorder=5)

    # Définir le vecteur vitesse moyenne entre M et M1.
    v_moy = [(M1[0] - M[0]) / dt, (M1[1] - M[1]) / dt]
    # Tracer en pointillés la ligne reliant M et le point M1 (en l'étendant bien au-delà) pour visualiser la direction de la trajectoire entre ces deux points.
    # definition d'un point Mx posé proportionnellement bien plus loin que M1
    Mx = [M[0] + 1e9 * (M1[0] - M[0]), M[1] + 1e9 * (M1[1] - M[1])]


    plt.plot([M[0], Mx[0]], [M[1], Mx[1]], 'k--', label='Direction de la trajectoire')
    # Restreindre les axes pour mieux visualiser la trajectoire et les points M, M1 et M0
    plt.xlim(-1, 3)
    plt.ylim(0, 5)  

    #  Faire une flèche épaisse et colorée en orange pour la différencier des autres éléments du graphe. Ajouter une légende pour expliquer ce que représente cette flèche.
    # Faire en sorte que la tête de flèche soit perpendiculaire à la trajectoire en M1 pour mieux visualiser la direction de la vitesse moyenne.
    plt.arrow(M[0], M[1], v_moy[0], v_moy[1],  head_length=0.2, head_width=0.1, linewidth=4, alpha = 0.8, fc='orange', ec='orange', label='Vitesse moyenne M -> M1', zorder = 10)
    
    plt.title('Trajectoire d\'une particule $M$')
    plt.gca().set_axisbelow(True)
    plt.xlabel('x(t)')
    plt.ylabel('y(t)')
    plt.grid()
    plt.legend()
    plt.show()

# Slider
interact(update, dt=(0.001, 2, 0.001))


Le module (ou norme) du vecteur vitesse peut aussi se calculer, de façon équivalente, via **l'abscisse curviligne $s$** de la trajectoire décrite par notre particule. Cette abscisse curviligne est un scalaire dépendant uniquement du temps et décrivant la distance parcourue, au temps $t$, par la particule puis l'origine jusqu'au point actuel. Il s'agit donc également de la longueur de l'arc décrit par la courbe $C$ entre le point $M_0$ et le pont $M$. La norme du vecteur vitesse peut ainsi s'exprimer comme

$$v(t) = \lim_{\Delta t \to 0} \frac{s(t+\Delta t) - s(t)}{\Delta t}$$

Enfin, si $\vec{T}$ désigne un vecteur unitaire ($\|\vec{T}\| = 1$) tangent à la courbe $C$ au point $M$, alors on peut exprimer le vecteur vitesse comme 

$$\overrightarrow{v}(t) = v(t) \vec{T} $$

#### 6.1.3 - Le vecteur accélération

Le **vecteur accélération** est, par définition, la dérivée du vecteur vitesse. On démontre en classe que ce vecteur accélération se décompose en deux contributions:

- une contribution tangentielle au mouvement
- une contribution normale au mouvement

$$\overrightarrow{a} = \vec{a}_T + \vec{a}_N = \frac{\textrm{d}^2 s}{\textrm{d}t^2} \vec{T} + \frac{V^2}{R} \vec{N} $$

où $R$ est le *rayon de courbure* de la courbe $C$ au point $M$ et $\vec{N}$ est le vecteur unitaire normal à $C$ au point $M$ (par convention, $\vec{n}$ pointe toujours vers l'intérieur)


Visualisons maintenant ce vecteur $\vec{a}$ ainsi que ses deux contributions $\vec{a}_T$ et $\vec{a}_N pour le mouvement étudié plus haut:


In [ ]:
# Calcul analytique du vecteur vitesse ainsi que du vecteur accélération à partir de la fonction de la trajectoire.
# Par hypothèse le champ de vitesse est donné par la fonction paramétrique u(t) = 1 et v(t) = 3(t-1)^2. Le champ d'accélération est donc donné par ax = 0 et ay = 6(t-1).

# Figure représentant la trajectoire de la particule M.
# Calcul de la vitesse à partir des composantes u et v

# On peut questionner les étudiants en demandant où la vitesse est la plus éleée, le rayon de courbure, l'accélération, etc.

# On insère par ailleurs ce graphe dans une fonction update afin de pouvoir faire varier le temps t et voir la particule se déplacer le long de la trajectoire, avec une couleur qui change en fonction de sa vitesse.

def update(t):
    x, y = trajectory(t)
    u = 1
    v = 3 * (t - 1)**2
    speed = np.sqrt(u**2 + v**2)
    
    plt.figure(figsize=(6, 4), dpi = 200)
    plt.plot(x_values, y_values, label='Trajectoire de M')
    plt.scatter(x, y, color=plt.cm.viridis(speed / np.max(speed)), s=100, label=f'M (t={t:.2f})', zorder = 10)
    # Flèche partant du point M et représentant le vecteur vitesse à ce point.
    plt.arrow(x, y, u, v, head_length=0.2, head_width=0.1, linewidth=2, alpha = 0.8, fc='orange', ec='orange', label='Vitesse instantanée', zorder = 10)

    # Flèche représentant le vecteur accélération à ce point. La flèche d'accélération est colorée en rouge et plus épaisse que celle de la vitesse pour la différencier visuellement.
    ax = 0
    ay = 6 * (t - 1)
    plt.arrow(x, y, ax, ay, head_length=0.2, head_width=0.1, linewidth=3, alpha = 0.8, fc='red', ec='red', label='Accélération instantanée', zorder = 10)

    # Calcul du rayon de courbure. On démontre (cf. cours de maths) que ce rayon de courbure est donné par R = (1+ f'(x)^2)^3/2 / |f''(x)|. On peut calculer f'(x) et f''(x) à partir de la fonction de la trajectoire.
    f_prime = 3 * (t - 1)**2
    f_double_prime = 6 * (t - 1)
    R = (1 + f_prime**2)**(3/2) / np.abs(f_double_prime)
    # Tracer un cercle de rayon R centré sur le point M pour visualiser le rayon de courbure à ce point. La couleur du cercle est choisie pour être différente de celle des flèches de vitesse et d'accélération pour la différencier visuellement.
    circle = plt.Circle((x, y), R, color='purple', fill=False, label='Rayon de courbure', zorder = 5)
    plt.gca().add_patch(circle) 

    # Définition du vecteur unitaire tangent à la trajectoire.
    T = [u / speed, v / speed]

    # Définition du vecteur unitaire normal à la trajectoire. Comme ce vecteur, en produit scalaire avec le tangente, doit être égal à zéro, on peut le définir comme suit (en tournant la tangente de 90 degrés dans le sens inverse des aiguilles d'une montre)
    # Cependant, par convention, on le définit toujours comme pointant vers l'intérieur de la courbure, c'est à dire dans le sens des aiguilles d'une montre. C'est pour cela que le vecteur normal est défini comme suit (en tournant la tangente de 90 degrés dans le sens des aiguilles d'une montre)
    # Dans ce cas particulier d'une fonction de la forme y = f(x), la direction du vecteur normal est donnée par le signe de f''(x). 
  
    N = [T[1] * (-np.sign(f_double_prime)), -T[0] * (-np.sign(f_double_prime)) ] 
    
    # Contribution normale à l'accélaration
    an = speed**2 / R
    # Vecteur normal correspondant
    an_vector = [an * N[0], an * N[1]]

    # Contribution tangentielle à l'accélaration. Calculé ici comme le vecteur accélération totale moins la contribution normale (parce que j'ai la flemme de calculer la dérivée de la vitesse pour trouver l'accélération tangentielle) :-)
    at_vector= [ax - an_vector[0], ay - an_vector[1]]
    at = np.sqrt(at_vector[0]**2 + at_vector[1]**2)
    

    # Plot des deux contributions. Ces contributions sont mises en pointillés pour les différencier de la flèche d'accélération totale. La contribution tangentielle est colorée en rouge et la contribution normale est colorée en violet pour les différencier visuellement.
    plt.arrow(x, y, at_vector[0], at_vector[1], head_length=0.2, head_width=0.1, linewidth=1, alpha = 0.8, fc='red', ec='red', label='Accélération tangentielle', zorder = 10, linestyle=':')
    plt.arrow(x, y, an_vector[0], an_vector[1], head_length=0.2, head_width=0.1, linewidth=1, alpha = 0.8, fc='purple', ec='purple', label='Accélération normale', zorder = 10, linestyle=':')
   

    plt.gca().set_aspect('equal', adjustable='datalim')
    plt.title('Trajectoire d\'une particule $M$ avec couleur selon la vitesse')
    plt.xlabel('x(t)')
    plt.ylabel('y(t)')
    plt.xlim(-1, 5)
    plt.ylim(-1, 5)  
    plt.grid()
    plt.legend()
    plt.show()

# Slider
interact(update, t= (0, 4, 0.01))


# On vérifie que la vitesse linéaire du point est faible au voisinage du minimum car c'est là que v(t) est le plus petit.



#### 6.1.4 - Et pour plusieurs particules?

L'étude de la mécanique des fluides a pour objectif de généraliser les concepts introduits à un ensemble de particules évoluant individuellement dans leur mouvement mais répondant à des contraintes collectives et globales.

Lorsque nous souhaitons étudier le mouvement des particules d'un fluides et caractériser un écoulement, deux approches (tout aussi valables et légitimes l'une que l'autre) sont possibles:

- L'approche (ou méthode ou description) **lagrangienne** (du nom de Joseph-Louis Lagrange, mathématicien et mécanicien italien, 1736-1813) prend le parti de suivre *dans leur mouvement* ces particules sur leurs propres trajectoires, et d'étudier les propriétés de ces particules (leur vitesse, mais aussi leur altitude, température, ...) au fur et à mesure que le temps passe. Avec cette approche, nous "suivons" le fluide de l'intérieur. Cette approche est particulièrement pertinente pour relier des concepts mécaniques à des observations prises, par exemple, à bord de ballons-sondes ou de bouées dérivantes qui en première approximation suivent le fluide dans son mouvement.
- L'approche (ou méthode ou description) **eulérienne** (du nom de Leonhard Euler, mathématicien et physicien suisse, 1707-1783) prend le parti d'associer à chaque point de l'espace un champ de vitesse et d'étudier les trajectoires qui en résultent. Cette approche est particulièrement pertinente pour relier des concepts mécaniques à des observations prises à grande échelle et de façon structurée spatialement, comme des mesures satellitaires.


A titre d'exemple: un conducteur roulant à 120 km/h sur une route de pente 3% et passe sous un pont à midi. Connaissant ces informations, il peut calculer son gain en altitude après une minute en connaissant sa propre vitesse, le temps écoulé, et la pente de la route (+60 m). Par ailleurs, un observateur situé au-dessus du pont peut facilement calculer que si l'autoroute baigne dans un champ de vitesse de 120 km/h et dans un champ d'élévation d'une pente de 3% vers les x-positifs, alors toute particule (dont celle qui passe sous le pont à midi) gagnera 30 m toutes les 30 secondes, soit 60 m par minute.

Dans la suite, nous travaillerons quasi-exclusivement dans le formalisme eulérien, car il est en pratique impossible d'identifier, suivre et modéliser l'évolution de particules individuelles. Il est cependant à noter qu'il existe de nombreux exemples dans lesquels l'approche lagrangienne est plus adaptée, flexible et commode, mais aussi moins coûteuse en temps de calcul, pour résoudre certains problèmes (back-tracking atmosphérique, déformation de nuages de polluants, ...). Le formalisme eulérien permet par ailleurs la visualisation globale et à grande échelle des dynamiques atmosphériques, ce qui est un de nos objectifs.





# 6.2 - Examen cinématique de quelques mouvements plans élémentaires


L'exemple utilisé jusqu'ici peut être repris pour introduire le concept de **champ de vitesse**. Nous avions défini la trajectoire d'une particule au moyen d'une courbe $C$ dont les coordonnées $x$ et $y$ sont paramétrées par la variable "temps" $t$: $x=x(t)$ et $y=y(t)$.

Voici à quoi ressemble le vecteur vitesse de la courbe ci-dessus, évalué à intervalles réguliers sur l'axe $x$.

In [ ]:
# Nous représentons le vecteur vitesse de la trajectoire ci-dessus, à intervalles réguliers x= 0 jusque x = 5, par pas de 0.5.
# On repart de la fonction "trajectory" introduite ci-dessus pour calculer les points de la trajectoire correspondant à ces valeurs de x, et ensuite calculer le vecteur vitesse à ces points pour les représenter par des flèches sur le graphe.

x_curve = np.arange(0, 5.5, 0.01)
y_curve = (x_curve - 1)**3 + 2
x_values = np.arange(0, 5.5, 1)
plt.figure(figsize=(6, 4), dpi = 200)
plt.plot(x_curve, y_curve, label='Trajectoire de M')
for x in x_values:
    t = x  # Comme x(t) = t, on peut directement utiliser x comme temps
    y = (t - 1)**3 + 2  # Calcul de y à partir de la fonction de la trajectoire
    u = 1  # Composante de la vitesse en x
    v = 3 * (t - 1)**2  # Composante de la vitesse en y
    speed = np.sqrt(u**2 + v**2)  # Calcul de la vitesse totale

    # Tracer le point correspondant à ce temps sur la trajectoire, avec une couleur selon la vitesse
    plt.scatter(x, y, color=plt.cm.viridis(speed / np.max(speed)), s=100, label=f'M (t={t:.2f})', zorder = 10)

    # Tracer une flèche représentant le vecteur vitesse à ce point. La longueur de la flèche est normalisée pour être visible sur le graphe.
    plt.arrow(x, y, u , v, head_length=0.2, head_width=0.1, linewidth=2, alpha = 0.8, fc='orange', ec='orange', label='Vitesse instantanée', zorder = 10)

plt.plot(x_values, (x_values - 1)**3 + 2, label='Trajectoire de M')
plt.title('Trajectoire d\'une particule $M$ avec vecteurs vitesse')
plt.xlabel('x(t)')
plt.ylabel('y(t)')
plt.xlim(-1, 6)
plt.ylim(-1, 30)
plt.grid()
plt.tight_layout()
plt.show()



Nous allons maintenant remplir l'ensemble du domaine de ce types de flèches en définissant un **champ de vitesse** comme un fonction vectorielle $\vec{u}$ à deux composantes ($u$ et $v$) dépendant chacune de deux variables, $x$ et $y$, et éventuellement du temps, $t$. 

Lorsque le champ de vitesse est indépendant du temps

$$\vec{u}(x,y) = \left(u(x,y), v(x,y)\right)$$

on dit de lui qu'il est "stationnaire": les particules sont en mouvement au sein de ce champ (pas statiques!) mais chaque fois qu'elle partent du même endroit, elles parcourent exactement les mêmes trajectoires.

A l'inverse, si le champ de vitesse dépend *explicitement* du temps 

$$\vec{u}(x,y, t) = (u(x,y, t), v(x,y, t))$$

alors ce champ évolue et les trajectoires se déforment.

Considérons quelques exemples simples, dans le plan horizontal, pour visualiser les dynamiques de divers champs de vitesse (stationnaires et non-stationnaires).



In [ ]:
# On efface toutes les variables de l'environnement pour repartir d'une page blanche.
# --- IGNORE ---
# On définit un domaine de dimensions 1 par 1 unité, avec une grille de 100 points en x et 100 points en y.
# On définit une variable temporelle courant sur 60 secondes, avec un pas de 0.1 seconde.

x = np.linspace(-2, 2, 100)
y = np.linspace(-2, 2, 100)
t = np.arange(0, 6, 0.01)

t_index = (0, len(t)-1, 1)



In [32]:
# On efface toutes les variables  pour repartir d'une page blanche.
# --- IGNORE ---

# Dans différents  champs de vent, on va maintenant placer un carré de particules (3 par 3) au centre du domaine.M On va ensuite faire évoluer ces particules dans le champ de vent en utilisant la méthode d'Euler pour calculer leur trajectoire. La méthode d'Euler consiste à calculer la position de chaque particule à l'instant t+dt en utilisant la position à l'instant t et la vitesse du vent à cette position. 
# On va faire évoluer ce carré en connaissant le champ analytique de vitesse.

# Notre première tâche est de créer une fonction qui prend en entrée la position initiale d'une particule
# et sort les positions x et y de cette particule à tous les moments ultérieurs en connaissant la fonction analytique du champ de vent.


# mode définit le type de mouvement
mode = 4

if mode == 1:
    mode_description = r"Pure translation rectiligine, champ stationnaire (u = 1, v = 1)"
    u = 0.5 * np.ones((len(t), len(x), len(y)))
    v = 0.5 * np.ones((len(t), len(x), len(y)))
elif mode == 2:
    mode_description = r"Pure translation, champ non-stationnaire (u = cos(2$\pi$ f t), v = sin(2$\pi$ f t))"
    freq = 0.5
    u = np.cos(2 * np.pi * freq * t)[:, np.newaxis, np.newaxis] * np.ones((len(t), len(x), len(y)))
    v = np.sin(2 * np.pi * freq * t)[:, np.newaxis, np.newaxis] * np.ones((len(t), len(x), len(y)))
elif mode == 3:
    mode_description = r"Pure divergence, champ stationnaire\n(u = 1 + x, v = 0)"
    u = (1 + x)[np.newaxis, np.newaxis, :] * np.ones((len(t), len(x), len(y)))
    v = np.zeros((len(t), len(x), len(y)))
elif mode == 4:
    mode_description = r"Pure rotation, champ stationnaire (u=-y, v = x)"
    # Le champ est u = -y et v = x
    # On veut 'broadcaster' le vecteur -y et x pour qu'il soit défini à tous les points de la grille (t, x, y). 
    u = np.zeros((len(t), len(x), len(y)))
    v = np.zeros((len(t), len(x), len(y)))
    for jx in range(len(x)):
        for jy in range(len(y)):
            for jt in range(len(t)):
                u[jt, jy, jx] = -y[jy]
                v[jt, jy, jx] = x[jx]
elif mode == 5:
    mode_description = r"Pur cisaillement, champ stationnaire (u=$\gamma_x$y,v = 0)"

    gammax = 1

    u = np.zeros((len(t), len(x), len(y)))
    for jx in range (len(x)):
        for jt in range (len(t)):
            u[jt, :, jx] = gammax * y[jy]
    v = np.zeros((len(t), len(x), len(y)))

elif mode == 6:
    mode_description = r"Cisaillement en x et en y, champ stationnaire (u =$\gamma_x$y, v=$\gamma_y$x )"
    u = np.zeros((len(t), len(x), len(y)))
    v = np.zeros((len(t), len(x), len(y)))

    # Toujours pris positifs
    gammax = 1
    gammay = 0.01

    for jt in range(len(t)):
        for jy in range(len(y)):
            u[jt, jy, :] = gammax * y[jy]
        for jx in range(len(x)):
            v[jt, :, jx] = gammay * x[jx]

    
# Wind speed
wspd = np.sqrt(u ** 2 + v ** 2)


def particle_trajectory(x0, y0, t, mode = 1):

    """
    mode = 1 correspond à un champ de vent constant (1, 1) partout
    mode = 2 correspond à un champ de vent oscillant dans le temps (cos(), sin()) partout
    mode = 3 correspond à un champ de vent exponentiel dans u = 1+x,
    mode = 4 correspond à un champ de vent tourbillonnant (-y, x)
    mode = 5 correspond à un champ de cisaillement pur
    mode = 6 correspond à un champ de cisaillement plus général

    La fonction retourne les positions x et y de la particule à tous les moments ultérieurs en connaissant la fonction analytique du champ de vent
       """
    x = np.zeros_like(t)
    y = np.zeros_like(t)
    x[0] = x0
    y[0] = y0
    
    if mode == 1:
        # Ceci suppose que le champ de vent est (1, 1) partout!!
        x[1:] = 0.5 * t[1:] + x0
        y[1:] = 0.5 * t[1:] + y0


    elif mode == 2:
        # Ceci suppose que le champ de vent est (cos(), (sin)) partout!!
        
        x[1:] = 1 / (2 * np.pi * freq) * np.sin(2 * np.pi * freq * t[1:]) + x0
        y[1:] = -1 / (2 * np.pi * freq) * np.cos(2 * np.pi * freq * t[1:]) + y0

    elif mode == 3:
        # Ceci suppose que le champ de vent est (1 +x, 0) partout!!
        # si dx/dt = 1 + x alors dx/(1+x) = dt et donc ln(1+x) = t + C et donc 1 + x = exp(t + C) et donc x = exp(t + C) - 1. 
        # En utilisant la condition initiale x(0) = x0, on trouve que C = ln(1 + x0) et donc x = exp(t) * (1 + x0) - 1. 
        x[1:] = np.exp(t[1:]) * (1 + x0) - 1
        y[1:] = y0

    elif mode ==4:  
        # Ceci suppose que le champ de vent est (-y, x) partout!!
        # Si dx/dt = -y et dy/dt = x, alors d^2x/dt^2 = -dy/dt = x et d^2y/dt^2 = dx/dt = -y
        # Les solutions de ces équations différentielles sont de la forme x = A*cos(t) + B*sin(t) et y = C*cos(t) + D*sin(t).
        # en imposant les conditions initiales et la relation u = -y et v = x, on trouve: 
        x[1:] = x0 * np.cos(t[1:]) - y0 * np.sin(t[1:])
        y[1:] = y0 * np.cos(t[1:]) + x0 * np.sin(t[1:])
    
    elif mode == 5:
        # dx/dt = gammax y et dy/dt = 0
        # Comme y = y0, alors x = t*y0*gammax + x0
        x[1:] = t[1:] * y0 * gammax + x0
        y[1:] =  y0

    elif mode == 6:
        # dx/ dt = gammax y
        # dy/dt  = gammay x
        # Donc d2x/dt2 = gammax gammay x avec gammax gammay positifs par hypotjhèse
        # Solution de type x(t) = A e^(sqrt(gammax gammay) t) + Be^(-sqrt(gammax gammay) t)
        # Or x(0) =A+B = x0
        # Par ailleurs y(t) a la solution de même forme C e^(sqrt(gammax gammay) t) + D e^(-sqrt(gammax gammay) t)
        # avec donc C + D = y0

        # Enfin dx / dt = A sqrt() e^(..t) - B sqrt() e^(-...t) = gammax C e^(..t) + gammax D e^(-sqrt(...t))
        # ce qui impose (pour tout t) A = gammax C et B = -gammax D

        # on déduit gammax C + B = x0
        # et        C - B / gammax = y0
        # donc C+C gamma x = x0 + y0 gammax --> C = (x0 + y0 gammax) / (1 + gammax)
        C = (x0 + y0 * gammax) / (1 + gammax)
        D = y0 - C
        B = x0 - C
        A = x0 - B

        x[1:] = A * np.exp(np.sqrt(gammax * gammay * t[1:])) + B * np.exp(-np.sqrt(gammax * gammay * t[1:]))
        y[1:] = C * np.exp(np.sqrt(gammax * gammay * t[1:])) + D * np.exp(-np.sqrt(gammax * gammay * t[1:]))

        
    return x, y

# Nous créons une liste de n traceurs. Chaque traceur est un dictionnaire contenant les positions x et y de la particule à tous les moments ultérieurs, 
# ainsi que les index des particules avec lesquelles elle sera reliée par une ligne (afin de visualiser un réseau)

tracers = list()

# Ajout de 9 traceurs en commençant par en haut à gauche, puis en allant vers la droite, puis à la ligne suivante, etc.
tracers += [
    {'x0': -0.3, 'y0': 0.3, 'connections': [1, 3]},  # Traceur 0
    {'x0': 0.0, 'y0': 0.3, 'connections': [0, 2, 4]},  # Traceur 1
    {'x0': 0.3, 'y0': 0.3, 'connections': [1, 5]},  # Traceur 2
    {'x0': -0.3, 'y0': 0.0, 'connections': [0, 4, 6]},  # Traceur 3
    {'x0': 0.0, 'y0': 0.0, 'connections': [1, 3, 5, 7]},  # Traceur 4
    {'x0': 0.3, 'y0': 0.0, 'connections': [2, 4, 8]},  # Traceur 5
    {'x0': -0.3, 'y0': -0.3, 'connections': [3, 7]},  # Traceur 6
    {'x0': 0.0, 'y0': -0.3, 'connections': [4, 6, 8]},  # Traceur 7
    {'x0': 0.3, 'y0': -0.3, 'connections': [5, 7]}   # Traceur 8
]


# Calculer les trajectoires de tous les traceurs

for tracer in tracers:
    tracer['x'], tracer['y'],  = particle_trajectory(tracer['x0'], tracer['y0'], t, mode = mode)

declutter = 5

# Créer une animation pour visualiser les trajectoires de tous les traceurs et les connexions entre eux. Les connexions sont représentées par des lignes reliant les traceurs connectés à chaque instant de temps.
def update(t_index):
    from IPython.display import clear_output
    clear_output(wait=True)     # évite d'empiler les figures

    plt.figure(figsize=(6, 5))
    
    #plt.contourf(x, y, wspd[t_index], cmap='viridis')

    #plt.quiver(x[::declutter], y[::declutter], 
    #           u[t_index, ::declutter, ::declutter] / wspd [t_index, ::declutter, ::declutter], 
    #           v[t_index, ::declutter, ::declutter] / wspd[t_index, ::declutter, ::declutter], 
    #           color = "darkgrey", scale=30)

    plt.quiver(x[::declutter], y[::declutter], 
               u[t_index, ::declutter, ::declutter] , 
               v[t_index, ::declutter, ::declutter] , 
               wspd[t_index, ::declutter, ::declutter], scale = 20, cmap = "viridis", )

    plt.colorbar(label='Module de la vitesse (m/s)')
    
    # Tracer les traceurs et les connexions
    for i, tracer in enumerate(tracers):
        plt.scatter(tracer['x'][t_index], tracer['y'][t_index], s=100, label=f'Tracer {i}', zorder = 10)
        for connection in tracer['connections']:
            plt.plot([tracer['x'][t_index], tracers[connection]['x'][t_index]], 
                     [tracer['y'][t_index], tracers[connection]['y'][t_index]], 
                     color='black', linestyle=':', alpha=0.5)

    plt.title(f"Champ et trajectoires: " + mode_description)
    plt.xlim(-1, 1)
    plt.ylim(-1, 1)
    plt.gca().set_aspect('equal', adjustable='datalim')

    plt.xlabel('x')
    plt.ylabel('y')
    #plt.legend()
    plt.tight_layout()
    plt.show()


# Slider pour faire varier le temps
interact(update, t_index=t_index)

interactive(children=(IntSlider(value=299, description='t_index', max=599), Output()), _dom_classes=('widget-i…

<function __main__.update(t_index)>

Qu'observons-nous? 
- Le champ en **pure translation** et **stationnaire** préserve les distances et les angles entre les particules. Le champ de vitesse, à un instant donné, est *indépendant* de $x$ et de $y$. L'ensemble des particules se déplace dans la même direction, dans le même sens, et à la même vitesse.

- Le champ en **pure translation** et **non-stationnaire** transporte le maillage sans le déformer. Même si l'ensemble décrit une trajectoire organisée circulairement, il n'y a aucune rotation en jeu!
- Le champ  **en pure divergence**  "distend" les mailles du réseau
- Le champ en **pure rotation** fait tourner sur lui-même le réseau de particules


Propriété fondamentale de mécanique des fluides: tout champ vectoriel $\vec{u}(x,y, t)$ quelconque, considéré à un instant donné $t$, est assimilable à un ensemble de combinaisons locales des champs fondamentaux introduits ci-dessus: translation, divergence, rotation, et cisaillement.




# Le coin des curieux(ses): décomposition du gradient de vitesse

C'est bien la manière dont le champ de vitesse varie dans l'espace qui définit le type de mouvements rencontrés en dynamique des fluides. Mathématiquement, ces variations s'étudient au moyen du **gradient du champ de vitesse**.

Soit un champ de vitesse bidimensionnel $\mathbf{U}(x,y) = (u(x,y), v(x,y))$.

Outre la translation (mouvement plan élémentaire), les différents types de mouvements locaux peuvent être obtenus en décomposant le gradient de vitesse, qui traduit donc la variation spatiale de $\mathbf{U}$.

Le gradient de vitesse est la matrice :

$$
\nabla \mathbf{U}
=
\begin{pmatrix}
\partial u / \partial x & \partial u / \partial y \\
\partial v / \partial x & \partial v / \partial y
\end{pmatrix}
$$

Comme $\nabla \mathbf{U}$ est une matrice $2 \times 2$, on peut la décomposer algébriquement en une partie symétrique et une partie antisymétrique :

$$
\nabla \mathbf{U} = \mathbf{S} + \mathbf{A}
$$

avec

$$
\mathbf{S} =
\frac{1}{2}
\left(
\nabla \mathbf{U}
+
(\nabla \mathbf{U})^T
\right)
$$

et

$$
\mathbf{A} =
\frac{1}{2}
\left(
\nabla \mathbf{U}
-
(\nabla \mathbf{U})^T
\right)
$$

On vérifie trivialement que :

\[
\mathbf{S} + \mathbf{A} = \nabla \mathbf{U}
\]

---

### Partie symétrique : la déformation

La matrice symétrique $\mathbf{S}$ s'écrit :

$$
\mathbf{S}
=
\begin{pmatrix}
\partial u / \partial x &
\frac{1}{2}
\left(
\partial u / \partial y + \partial v / \partial x
\right)
\\
\frac{1}{2}
\left(
\partial u / \partial y + \partial v / \partial x
\right)
&
\partial v / \partial y
\end{pmatrix}
$$

Elle décrit la **déformation locale**, et est composée de deux contributions :

- la **divergence** (les termes diagonaux) :

$$
\nabla \cdot \mathbf{U}
=
\frac{\partial u}{\partial x}
+
\frac{\partial v}{\partial y}
$$

qui correspond à une dilatation ou compression locale causant une variation d'aire ;

- le **cisaillement pur** (les termes hors diagonale), qui correspond à une déformation sans rotation.

---

### Partie antisymétrique : rotation solide

La matrice antisymétrique $\mathbf{A}$ s'écrit :

$$
\mathbf{A}
=
\begin{pmatrix}
0 &
\frac{1}{2}
\left(
\partial u / \partial y - \partial v / \partial x
\right)
\\
\frac{1}{2}
\left(
\partial v / \partial x - \partial u / \partial y
\right)
&
0
\end{pmatrix}
$$

Elle décrit la **rotation solide locale** du fluide.

---

### Résumé

Le mouvement local d'un fluide peut être décomposé en :

$$
\text{Mouvement}
=
\underbrace{\text{Déformation}}_{\text{divergence + cisaillement pur}}
+
\text{Rotation solide}
$$

---

### Remarque importante : cisaillement simple *vs.* cisaillement pur

Un **cisaillement simple** :

$$
u = \gamma y, \qquad v = 0
$$

donne :

$$
\nabla \mathbf{U}
=
\begin{pmatrix}
0 & \gamma \\
0 & 0
\end{pmatrix}
$$

qui se décompose donc en

$$
\mathbf{A}
=
\begin{pmatrix}
0 & \gamma/2 \\
\gamma/2 & 0
\end{pmatrix}
\qquad
\mathbf{B}
=
\begin{pmatrix}
0 & \gamma/2 \\
-\gamma/2 & 0
\end{pmatrix}
$$

Donc le cisaillement simple contient donc :

- du cisaillement pur
- et de la rotation solide

---

En revanche, un **cisaillement pur** :

$$
u = \frac{\gamma}{2} y,
\qquad
v = \frac{\gamma}{2} x
$$

donne :

$$
\mathbf{A}
=
\begin{pmatrix}
0 & \gamma/2 \\
\gamma/2 & 0
\end{pmatrix}
\qquad
\mathbf{B} = 0
$$

et n'est donc associé à aucune rotation solide.

---

### Remarque : déformation sans divergence ni cisaillement

Un champ :

$$
u = \gamma x,
\qquad
v = -\gamma y
$$

donne :

$$
\nabla \mathbf{U}
=
\begin{pmatrix}
\gamma & 0 \\
0 & -\gamma
\end{pmatrix}
$$

Ce mouvement correspond à une **déformation pure sans divergence** et **sans rotation**.


## Questions

- Soit une particule décrivant une courbe $C$ paramétrée par son abscisse curviligne $s(t)$. En introduisant les éléments cinématiques et géométriques adéquats, établir l'expression du vecteur accélération de la particule et proposer une décomposition de ce vecteur en des composantes facilement interprétables.
- Définir la notion de *rayon de courbure* au point $M$ d'une courbe $C$
- En mécanique des fluides, qu'est-ce qui distingue l'approche lagrangienne de l'approche eulérienne? 
- Quelles sont les quatre grands types de mouvements dans un fluide géophysique? 

In [35]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

# Function to update the plot
def update(x):
    plt.figure(figsize=(6,4))
    
    # Create curve
    t = np.linspace(0, 10, 400)
    y = np.sin(t)
    plt.plot(t, y)
    
    # Moving point
    plt.scatter(x, np.sin(x), color='red', s=100)
    
    plt.xlim(0, 10)
    plt.ylim(-1.5, 1.5)
    plt.grid()
    plt.show()

# Slider
interact(update, x=(0, 10, 0.1))


interactive(children=(FloatSlider(value=5.0, description='x', max=10.0), Output()), _dom_classes=('widget-inte…

<function __main__.update(x)>